# Data Exploration Report
> Profiling raw source files for the Special Olympics ETL pipeline.

This notebook reproduces every finding from sections 1–3 of
`docs/data_exploration.md` using executable pandas code.

In [1]:
import sys
import os

# Add project root to sys.path for imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "data")):
    PROJECT_ROOT = os.getcwd()  # Already at project root
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from collections import Counter
from src.explore import DataProfiler
from src.utils.data_loader import DataLoader

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
loader = DataLoader(RAW_DIR)
profiler = DataProfiler(RAW_DIR)

---
## 1. Certifications File
(`Thomas More Data Certifications.xlsx`)

### Structure

In [2]:
# Load using DataLoader (cached)
cert_df = loader.load_certifications()

# Quick structure via DataProfiler
cert_profile = profiler.profile_file("Thomas More Data Certifications.xlsx")
sheet_info = cert_profile["sheets"]["Sheet1"]

print(f"Sheets:   1 (Sheet1)")
print(f"Shape:    {sheet_info['row_count']:,} rows \u00d7 {len(sheet_info['columns'])} columns")
print(f"\nColumns ({len(sheet_info['columns'])}):")
for c in sheet_info["columns"]:
    print(f"  \u2022 {c}")

Sheets:   1 (Sheet1)
Shape:    21,001 rows × 10 columns

Columns (10):
  • Club
  • Code
  • Person type
  • Gender
  • DOB
  • Age
  • Mental Handicap (SOB has this certificate)
  • Parents Consent (SOB has this certificate)
  • HAP (SOB has this certificate)
  • Unified Partner (SOB has this certificate)


In [3]:
print("Column dtypes:\n")
for col, dtype in cert_df.dtypes.items():
    print(f"  {col:<50s} {dtype}")

Column dtypes:

  Club                                               object
  Code                                               object
  Person type                                        object
  Gender                                             object
  DOB                                                datetime64[ns]
  Age                                                float64
  Mental Handicap (SOB has this certificate)         float64
  Parents Consent (SOB has this certificate)         float64
  HAP (SOB has this certificate)                     float64
  Unified Partner (SOB has this certificate)         float64


### Column Analysis

#### Code (Primary Key)

In [4]:
codes = cert_df["Code"]

print(f"Non-null values:           {codes.notna().sum():,}")
print(f"Unique (among non-null):   {codes.dropna().nunique():,}")
print(f"Missing (NaN):             {codes.isna().sum():,}  "
      f"({codes.isna().mean():.2%})")

# Check for duplicates among non-null codes
dup_codes = codes.dropna()[codes.dropna().duplicated()]
print(f"Duplicates among non-null: {len(dup_codes)}")

Non-null values:           20,221
Unique (among non-null):   20,221
Missing (NaN):             780  (3.71%)
Duplicates among non-null: 0


In [5]:
# Code format inspection
sample_codes = codes.dropna().head(5).tolist()
print("Sample codes:", sample_codes)

code_lengths = codes.dropna().str.len().value_counts()
print(f"\nCode lengths:\n{code_lengths.to_string()}")
print(f"\nAll 16-char: {(codes.dropna().str.len() == 16).all()}")
print(f"All alphanumeric uppercase: "
      f"{codes.dropna().str.match(r'^[A-Z0-9]+$').all()}")

Sample codes: ['3B1D6O9Z08W7IOZG', '4PQUISSS7HAL3BVW', '5YFHEPFDCRU2M1PA', '7NHXE4UAQUNMGVVR', 'B3HQ4KQU4M8DR0B6']

Code lengths:
Code
16    20221

All 16-char: True
All alphanumeric uppercase: True


#### Gender

In [6]:
gender = cert_df["Gender"]
gender_counts = gender.value_counts(dropna=False).sort_values(ascending=False)
total = len(cert_df)

print("Gender distribution:\n")
print(f"  {'Value':<8} {'Count':>7}  {'%':>7}")
print(f"  {'-'*8} {'-'*7}  {'-'*7}")
for val, cnt in gender_counts.items():
    label = "NaN" if pd.isna(val) else str(val)
    print(f"  {label:<8} {cnt:>7,}  {cnt/total:>7.2%}")

print(f"\nDistinct non-null values: {gender.dropna().nunique()}")

Gender distribution:

  Value      Count        %
  -------- -------  -------
  M         12,530   59.66%
  F          7,640   36.38%
  NaN          780    3.71%
  U             51    0.24%

Distinct non-null values: 3


In [7]:
# U-gender breakdown by Person type
u_mask = cert_df["Gender"] == "U"
print(f"Gender='U' rows ({u_mask.sum()}) by Person type:\n")
print(cert_df.loc[u_mask, "Person type"].value_counts().to_string())

Gender='U' rows (51) by Person type:

Person type
Unified Partner    22
Athlete            17
Coach               9
Volunteer           2
A-HOD               1


In [8]:
# Confirm no Male/Female long-form variants
print("All unique gender values:", sorted(gender.dropna().unique()))

All unique gender values: ['F', 'M', 'U']


#### DOB (Date of Birth)

In [9]:
dob = cert_df["DOB"]

print(f"Dtype:   {dob.dtype}")
print(f"Missing: {dob.isna().sum():,}  ({dob.isna().mean():.2%})")
print(f"Range:   {dob.min()} → {dob.max()}")

# How many real records (non-empty-row) lack a DOB?
empty_row_mask = cert_df.drop(columns="Code", errors="ignore").isna().all(axis=1) & cert_df["Code"].isna()
empty_rows = empty_row_mask.sum()
real_missing_dob = dob.isna().sum() - empty_rows
print(f"\nEmpty rows (all NaN):          {empty_rows:,}")
print(f"Real records missing DOB:      {real_missing_dob:,}  "
      f"(~{real_missing_dob / total:.1%} of total)")

Dtype:   datetime64[ns]
Missing: 1,990  (9.48%)
Range:   1900-01-02 00:00:00 → 2020-12-05 00:00:00

Empty rows (all NaN):          780
Real records missing DOB:      1,210  (~5.8% of total)


In [10]:
# Decade distribution
dob_valid = dob.dropna()
decades = (dob_valid.dt.year // 10 * 10).value_counts().sort_index()

print("Decade distribution:\n")
print(f"  {'Decade':<8} {'Count':>6}")
print(f"  {'-'*8} {'-'*6}")
for decade, count in decades.items():
    print(f"  {int(decade):<8} {count:>6,}")

Decade distribution:

  Decade    Count
  -------- ------
  1900          6
  1920          1
  1930         14
  1940        209
  1950      1,171
  1960      2,584
  1970      2,939
  1980      3,622
  1990      4,515
  2000      3,300
  2010        648
  2020          2


In [11]:
# Suspicious / sentinel dates
print("Suspicious dates:\n")

sentinel_mask = dob == pd.Timestamp("1900-01-02")
print(f"DOB = 1900-01-02 (sentinel): {sentinel_mask.sum()} rows")
if sentinel_mask.any():
    print(f"  Ages of these rows: "
          f"{cert_df.loc[sentinel_mask, 'Age'].unique()}")

very_old = cert_df[cert_df["Age"] > 100]
print(f"\nAge > 100: {len(very_old)} rows")
print(very_old[["Code", "DOB", "Age", "Person type"]].to_string(index=False))

young_2020 = cert_df[dob.dt.year == 2020] if dob.notna().any() else pd.DataFrame()
print(f"\nBorn in 2020: {len(young_2020)} rows")

Suspicious dates:

DOB = 1900-01-02 (sentinel): 6 rows
  Ages of these rows: [125.]

Age > 100: 7 rows
            Code        DOB   Age     Person type
UZ6DFP7E2K27SES6 1900-01-02 125.0         Athlete
2BOACQT165QP8L8D 1900-01-02 125.0         Athlete
6X5MOP83JR2WSX9Q 1900-01-02 125.0         Athlete
93M9PTH5MT8ZU59A 1900-01-02 125.0 Unified Partner
NHPN9TH6EJZBURHK 1900-01-02 125.0 Unified Partner
XHF3P9GMN6EZXZT8 1900-01-02 125.0         Athlete
VB58Q4UETVTR5YB6 1921-10-24 104.0         Athlete

Born in 2020: 2 rows


#### Age

In [12]:
age = cert_df["Age"]

print(f"Missing:  {age.isna().sum():,}")
print(f"Range:    {age.min():.0f} – {age.max():.0f}")
print(f"Mean:     {age.mean():.1f}")
print(f"Median:   {age.median():.0f}")

Missing:  780
Range:    0 – 125
Mean:     37.7
Median:   36


In [13]:
# Age = 0 analysis
age_zero = cert_df[age == 0]
print(f"Age = 0: {len(age_zero):,} rows\n")
print("DOB status of Age=0 rows:")
print(f"  DOB is NaT: {age_zero['DOB'].isna().sum()}")
print(f"  DOB present: {age_zero['DOB'].notna().sum()}")

print(f"\nPerson types of Age=0 rows:")
print(age_zero["Person type"].value_counts().to_string())

Age = 0: 1,210 rows

DOB status of Age=0 rows:
  DOB is NaT: 1210
  DOB present: 0

Person types of Age=0 rows:
Person type
Coach              754
Athlete            271
Unified Partner    158
Staff               17
Volunteer            5
Manager              1
Security             1
Head Coach           1
A-HOD                1
AS-Staff             1


In [14]:
# Age > 100
print(f"Age > 100: {(age > 100).sum()} rows "
      f"(6 sentinel-date + 1 born 1921)")

Age > 100: 7 rows (6 sentinel-date + 1 born 1921)


#### Person Type

In [15]:
pt = cert_df["Person type"]
pt_counts = pt.value_counts(dropna=False).sort_values(ascending=False)

print("Person type distribution:\n")
print(f"  {'Value':<18} {'Count':>7}  {'%':>7}")
print(f"  {'-'*18} {'-'*7}  {'-'*7}")
for val, cnt in pt_counts.items():
    label = "NaN" if pd.isna(val) else str(val)
    print(f"  {label:<18} {cnt:>7,}  {cnt/total:>7.2%}")

print(f"\nDistinct types (excl NaN): {pt.dropna().nunique()}")

Person type distribution:

  Value                Count        %
  ------------------ -------  -------
  Athlete             15,864   75.54%
  Coach                3,801   18.10%
  NaN                    780    3.71%
  Unified Partner        516    2.46%
  Staff                   19    0.09%
  Volunteer                6    0.03%
  Head Coach               3    0.01%
  VIP                      2    0.01%
  Medical                  2    0.01%
  Security                 2    0.01%
  A-HOD                    2    0.01%
  Manager                  1    0.00%
  Family member            1    0.00%
  Media                    1    0.00%
  AS-Staff                 1    0.00%

Distinct types (excl NaN): 14


#### Club

In [16]:
club = cert_df["Club"]

print(f"Unique clubs:  {club.dropna().nunique()}")
print(f"Missing:       {club.isna().sum():,}")

Unique clubs:  434
Missing:       780


In [17]:
# Largest clubs
club_counts = club.value_counts()
print("Top 10 clubs:\n")
print(club_counts.head(10).to_string())

Top 10 clubs:

Club
General                  7310
SOMIVAL VZW               367
CENTRE REINE FABIOLA      259
LES OURSONS               234
DE HOGE KOUTER            226
EMBARQUEMENT IMMEDIAT     202
DE MOLENKREKELS           192
LA PILERIE                182
ZONNELIED                 181
CENTRE CERFONTAINE        180


In [18]:
# "General" club analysis
general_count = club_counts.get("General", 0)
print(f"\n'General' members: {general_count:,}  "
      f"({general_count / total:.1%} of all rows)")
print("→ Appears to be a catch-all/default assignment, not a real club.")


'General' members: 7,310  (34.8% of all rows)
→ Appears to be a catch-all/default assignment, not a real club.


In [19]:
# Club name character examples
print("\nSample club names with special characters:")
special = club.dropna().unique()
for name in sorted(special):
    if any(c for c in name if ord(c) > 127):
        print(f"  • {name}")
    if name == "HAGEN (Germany)":
        print(f"  • {name}  [foreign club]")


Sample club names with special characters:
  • 16 ÂMES
  • FOYER DE THÉMIS
  • HAGEN (Germany)  [foreign club]
  • L'ENVOLÉE
  • LA CLAIRIÈRE
  • LE CHAT BOTTÉ
  • LE JARDIN DES FÉES
  • LE NIDVELLES° (LE COLOMBIER 2)
  • LES 3 FÉES
  • LES BLÉS D’OR
  • PÉRUWELZ RAINBOW SPORTS
  • RESIDENCE ÉLYSÉE
  • VISÉ - CENTRE DE CERFONTAINE
  • ÉTOILE MARCHOISE


#### Certificate Columns

In [20]:
cert_cols = [
    "Mental Handicap (SOB has this certificate)",
    "Parents Consent (SOB has this certificate)",
    "HAP (SOB has this certificate)",
    "Unified Partner (SOB has this certificate)",
]

short_names = ["Mental Handicap", "Parents Consent", "HAP", "Unified Partner"]

print("Certificate column breakdown:\n")
print(f"  {'Certificate':<18} {'Has (1)':>8} {'No (0)':>8} {'NaN':>8}")
print(f"  {'-'*18} {'-'*8} {'-'*8} {'-'*8}")
for short, col in zip(short_names, cert_cols):
    has = (cert_df[col] == 1).sum()
    no = (cert_df[col] == 0).sum()
    na = cert_df[col].isna().sum()
    print(f"  {short:<18} {has:>8,} {no:>8,} {na:>8,}")

Certificate column breakdown:

  Certificate         Has (1)   No (0)      NaN
  ------------------ -------- -------- --------
  Mental Handicap       9,795       18   11,188
  Parents Consent       6,616       27   14,358
  HAP                   5,526        1   15,474
  Unified Partner         183        0   20,818


In [21]:
# Unified Partner certificate cross-tab with Person type
up_col = "Unified Partner (SOB has this certificate)"
up_mask = cert_df[up_col] == 1
print(f"People with Unified Partner certificate ({up_mask.sum()}):\n")
print(cert_df.loc[up_mask, "Person type"].value_counts().to_string())

People with Unified Partner certificate (183):

Person type
Unified Partner    143
Coach               39
Athlete              1


### Data Quality: Missing Values Summary

In [22]:
print("Missing values per column:\n")
missing = cert_df.isna().sum()
print(f"  {'Column':<50} {'Missing':>7} {'%':>7}")
print(f"  {'-'*50} {'-'*7} {'-'*7}")
for col, cnt in missing.items():
    print(f"  {col:<50} {cnt:>7,} {cnt/total:>7.2%}")

Missing values per column:

  Column                                             Missing       %
  -------------------------------------------------- ------- -------
  Club                                                   780   3.71%
  Code                                                   780   3.71%
  Person type                                            780   3.71%
  Gender                                                 780   3.71%
  DOB                                                  1,990   9.48%
  Age                                                    780   3.71%
  Mental Handicap (SOB has this certificate)          11,188  53.27%
  Parents Consent (SOB has this certificate)          14,358  68.37%
  HAP (SOB has this certificate)                      15,474  73.68%
  Unified Partner (SOB has this certificate)          20,818  99.13%


### Data Quality: Empty Rows

In [23]:
# Detect empty rows using DataProfiler duplicate detection
# (empty rows = all columns NaN = duplicated on no meaningful key)
empty_mask = cert_df.isna().all(axis=1)
empty_count = empty_mask.sum()
print(f"Completely empty rows: {empty_count} ({empty_count / len(cert_df) * 100:.1f}%)")

# Also check for duplicate Codes using DataProfiler
code_dups = DataProfiler.detect_duplicates(cert_df.dropna(subset=["Code"]), ["Code"])
print(f"Duplicate Codes (among non-null): {len(code_dups)} rows")

Completely empty rows: 780 (3.7%)
Duplicate Codes (among non-null): 0 rows


### Key Observations

1. **Code is a reliable primary key** — 16-char alphanumeric, unique across all
   non-null rows. Safe to use as `AthleteID`.
2. **No gender standardization needed** — already `M`/`F` (no `Male`/`Female`
   variants). `U` is the only extra value (51 rows).
3. **DOB is already datetime** — no parsing required, but sentinel values
   (`1900-01-02`) and ~1,210 real missing values need cleanup.
4. **"General" club dominates** — 34.8% of records. Needs special handling
   in geography dimension and fuzzy matching.
5. **Coaches lack DOBs** — Age = 0 is a red flag. ETL should null-out Age
   when DOB is missing.
6. **Certificate columns are sparse** — useful but NaN-heavy.

---
## 2. Clubs File
(`Thomas More Data Clubs.xlsx`)

### Structure

In [24]:
# Load using DataLoader (cached)
clubs_df = loader.load_clubs()

# Quick structure via DataProfiler
clubs_profile = profiler.profile_file("Thomas More Data Clubs.xlsx")
sheet_info = clubs_profile["sheets"]["Sheet1"]

print(f"Sheets:   {len(clubs_profile['sheets'])} (Sheet1)")
print(f"Shape:    {sheet_info['row_count']} rows \u00d7 {len(sheet_info['columns'])} columns")
print(f"\nColumns ({len(sheet_info['columns'])}):")
for c in sheet_info["columns"]:
    print(f"  \u2022 {c}")

Sheets:   1 (Sheet1)
Shape:    436 rows × 17 columns

Columns (17):
  • Group number
  • Name
  • Primary language
  • Address (Street and Number)
  • Zipcode
  • City
  • Province
  • Country
  • Participation Games 2015
  • Participation Games 2016
  • Participation Games 2017
  • Participation Games 2018
  • Participation Games 2019
  • Participation Games 2022
  • Participation Games 2023
  • Participation Games 2024
  • Participation Games 2025


In [25]:
print("Column dtypes:\n")
for col, dtype in clubs_df.dtypes.items():
    print(f"  {col:<35s} {dtype}")

Column dtypes:

  Group number                        int64
  Name                                object
  Primary language                    object
  Address (Street and Number)         object
  Zipcode                             float64
  City                                object
  Province                            object
  Country                             object
  Participation Games 2015            float64
  Participation Games 2016            bool
  Participation Games 2017            bool
  Participation Games 2018            bool
  Participation Games 2019            bool
  Participation Games 2022            float64
  Participation Games 2023            float64
  Participation Games 2024            float64
  Participation Games 2025            float64


### Column Analysis

#### Group Number (primary key candidate)

In [26]:
gn = clubs_df["Group number"]

print(f"Unique count: {gn.nunique()} (= row count → fully unique: {gn.nunique() == len(clubs_df)})")
print(f"Missing:      {gn.isna().sum()}")
print(f"Range:        {gn.min()} – {gn.max()}")

# Count gaps in sequence
full_range = set(range(int(gn.min()), int(gn.max()) + 1))
actual = set(gn.dropna().astype(int))
gaps = full_range - actual
print(f"Gaps in sequence: {len(gaps)}")

Unique count: 436 (= row count → fully unique: True)
Missing:      0
Range:        100 – 788
Gaps in sequence: 253


#### Name (join key with Results)

In [27]:
name = clubs_df["Name"]

print(f"Unique count: {name.nunique()} (= row count: {name.nunique() == len(clubs_df)})")
print(f"Missing:      {name.isna().sum()}")
lengths = name.str.len()
print(f"Length range:  {lengths.min()}–{lengths.max()} characters (mean {lengths.mean():.1f})")

Unique count: 436 (= row count: True)
Missing:      0
Length range:  3–41 characters (mean 14.3)


#### Province

In [28]:
prov = clubs_df["Province"]

print(f"Unique values: {prov.nunique()}")
print(f"Missing:       {prov.isna().sum()} ({prov.isna().mean():.1%})")

print("\nAll province values (with counts):\n")
print(prov.value_counts().to_string())

Unique values: 24
Missing:       6 (1.4%)

All province values (with counts):

Province
Hainaut              76
Antwerpen            58
Oost-Vlaanderen      48
West-Vlaanderen      44
Liège                39
Limburg              35
Brussel/Bruxelles    29
Vlaams Brabant       28
Brabant Wallon       19
Namur                19
Luxembourg           12
Bruxelles             5
ANTWERPEN             3
HAINAUT               2
Vlaams-Brabant        2
Oost Vlaanderen       2
Antwerpen             2
Babant Wallon         1
Brabant-Wallon        1
Wallonie              1
BRUSSEL               1
West-Vlaanderen       1
West- Vlaanderen      1
Belgie                1


In [29]:
# Highlight spelling/formatting inconsistencies
print("Province variant groups:\n")

# Group by lowercase+stripped for comparison
prov_clean = prov.dropna().str.strip()
variant_map = {}
for v in prov_clean.unique():
    key = v.lower().replace("-", " ").replace("  ", " ").strip()
    variant_map.setdefault(key, []).append(v)

for key, variants in sorted(variant_map.items()):
    if len(variants) > 1:
        counts = {v: (prov_clean == v).sum() for v in variants}
        print(f"  {key}:")
        for v, c in counts.items():
            print(f"    '{v}' ({c})")

Province variant groups:

  antwerpen:
    'Antwerpen' (60)
    'ANTWERPEN' (3)
  brabant wallon:
    'Brabant Wallon' (19)
    'Brabant-Wallon' (1)
  hainaut:
    'Hainaut' (76)
    'HAINAUT' (2)
  oost vlaanderen:
    'Oost-Vlaanderen' (48)
    'Oost Vlaanderen' (2)
  vlaams brabant:
    'Vlaams Brabant' (28)
    'Vlaams-Brabant' (2)
  west vlaanderen:
    'West-Vlaanderen' (45)
    'West- Vlaanderen' (1)


In [30]:
# Erroneous province values
print("Erroneous values (not a Belgian province):")
for val in ["Belgie", "Wallonie"]:
    cnt = (prov == val).sum()
    if cnt > 0:
        print(f"  '{val}': {cnt} row(s)")

Erroneous values (not a Belgian province):
  'Belgie': 1 row(s)
  'Wallonie': 1 row(s)


#### Country

In [31]:
country = clubs_df["Country"]

print(f"Unique values: {country.nunique()}")
print(f"Missing:       {country.isna().sum()} ({country.isna().mean():.1%})")

print("\nAll country variants:\n")
country_counts = country.value_counts(dropna=False).sort_values(ascending=False)
for val, cnt in country_counts.items():
    label = "NaN" if pd.isna(val) else str(val)
    print(f"  {label:<20} {cnt:>4}")

Unique values: 14
Missing:       74 (17.0%)

All country variants:

  BELGIQUE              121
  BELGIË                 99
  NaN                    74
  Belgique               42
  Belgie                 38
  BELGIE                 25
  Belgium                25
  België                  5
  Luxembourg              1
  belgique                1
  WAREGEM                 1
  Belguim                 1
  Belgïe                  1
  West-Vlaanderen         1
  Belgïum                 1


In [32]:
# Flag erroneous country values
print("\nErroneous country values:")
print("  'WAREGEM' (1) — city, not a country")
print("  'West-Vlaanderen' (1) — province (row 433, Province/Country swapped)")
print("  'Luxembourg' (1) — only non-Belgium country")


Erroneous country values:
  'WAREGEM' (1) — city, not a country
  'West-Vlaanderen' (1) — province (row 433, Province/Country swapped)
  'Luxembourg' (1) — only non-Belgium country


#### City

In [33]:
city = clubs_df["City"]

print(f"Unique cities:  {city.nunique()}")
print(f"Missing:        {city.isna().sum()}")

# Casing analysis
all_caps = city.str.match(r"^[^a-z]*$").sum()
mixed = len(city) - all_caps
print(f"\nALL CAPS cities: {all_caps} ({all_caps/len(city):.1%})")
print(f"Mixed case:      {mixed} ({mixed/len(city):.1%})")

Unique cities:  345
Missing:        0

ALL CAPS cities: 277 (63.5%)
Mixed case:      159 (36.5%)


In [34]:
# Duplicate cities with different casing
print("Cities with casing variants:\n")
city_lower = city.str.lower()
for name_lower in sorted(city_lower.unique()):
    originals = city[city_lower == name_lower].unique()
    if len(originals) > 1:
        detail = ", ".join(
            f"'{o}' ({(city == o).sum()})" for o in originals
        )
        print(f"  {detail}")

Cities with casing variants:

  'BAZEL' (1), 'Bazel' (1)
  'Berchem' (2), 'BERCHEM' (1)
  'BOECHOUT' (1), 'Boechout' (1)
  'BRAINE L'ALLEUD' (1), 'Braine L'Alleud' (1)
  'BRASSCHAAT' (1), 'Brasschaat' (2)
  'BRUGGE' (1), 'Brugge' (1)
  'DEINZE' (2), 'Deinze' (1)
  'ESTAIMPUIS' (1), 'Estaimpuis' (1)
  'Genk' (4), 'GENK' (3)
  'GENT' (3), 'Gent' (2)
  'Grâce-Hollogne' (1), 'GRÂCE-HOLLOGNE' (1)
  'HASSELT' (2), 'Hasselt' (2)
  'HERENT' (1), 'Herent' (1)
  'HERENTALS' (1), 'Herentals' (1)
  'KLUISBERGEN' (1), 'Kluisbergen' (1)
  'KORTRIJK' (4), 'Kortrijk' (1)
  'LA LOUVIÈRE' (2), 'La Louvière' (1)
  'LAAKDAL' (1), 'Laakdal' (1)
  'Leuven' (1), 'LEUVEN' (2)
  'LOKEREN' (1), 'Lokeren' (1)
  'LOMMEL' (2), 'Lommel' (1)
  'LUMMEN' (1), 'Lummen' (2)
  'Marcinelle' (1), 'MARCINELLE' (1)
  'MECHELEN' (1), 'Mechelen' (3)
  'MERKSEM' (2), 'Merksem' (2)
  'MIGNAULT' (1), 'Mignault' (1)
  'Mons' (3), 'MONS' (1)
  'MOUSCRON' (3), 'Mouscron' (1)
  'NAMUR' (4), 'Namur' (1)
  'NIVELLES' (2), 'Nivelles' (4

In [35]:
# Top cities
print("\nTop 10 cities by club count:\n")
print(city.value_counts().head(10).to_string())


Top 10 cities by club count:

City
BRUXELLES    21
GEEL          6
Liège         4
Genk          4
NAMUR         4
KORTRIJK      4
Nivelles      4
GENK          3
GENT          3
Mons          3


#### Primary Language

In [36]:
lang = clubs_df["Primary language"]

print(f"Unique values: {lang.nunique()}")
print(f"Missing:       {lang.isna().sum()}")
print(f"\nDistribution:")
for val, cnt in lang.value_counts().items():
    print(f"  {val}: {cnt} ({cnt/len(clubs_df):.1%})")
print("\n→ Clean column — no issues.")

Unique values: 2
Missing:       0

Distribution:
  Dutch: 229 (52.5%)
  French: 207 (47.5%)

→ Clean column — no issues.


#### Zipcode

In [37]:
zc = clubs_df["Zipcode"]

print(f"Dtype:    {zc.dtype}")
print(f"Missing:  {zc.isna().sum()} ({zc.isna().mean():.1%})")
print(f"Range:    {zc.min():.0f} – {zc.max():.0f}")

Dtype:    float64
Missing:  3 (0.7%)
Range:    1000 – 29900


In [38]:
# Outlier detection
outlier_mask = zc > 9999
print(f"Zipcodes > 9999 (Belgian max): {outlier_mask.sum()}")
if outlier_mask.any():
    print(clubs_df.loc[outlier_mask, ["Group number", "Name", "Zipcode", "City"]]
          .to_string(index=False))
    print("→ Likely 29900 should be 2990 (Wuustwezel)")

Zipcodes > 9999 (Belgian max): 1
 Group number              Name  Zipcode       City
          787 ATHLETES FOR HOPE  29900.0 WUUSTWEZEL
→ Likely 29900 should be 2990 (Wuustwezel)


#### Row 433 — Province / Country Swap

In [39]:
# Province/Country values are swapped for Group 786
row_786 = clubs_df[clubs_df["Group number"] == 786]
print("Group 786 (KHC SAINT-GEORGES):\n")
print(row_786[["Group number", "Name", "Province", "Country"]].to_string(index=False))
print("\n→ Province and Country are swapped here.")
print("   Province shows 'Belgie' (a country), Country shows 'West-Vlaanderen' (a province).")

Group 786 (KHC SAINT-GEORGES):

 Group number              Name Province         Country
          786 KHC SAINT-GEORGES   Belgie West-Vlaanderen

→ Province and Country are swapped here.
   Province shows 'Belgie' (a country), Country shows 'West-Vlaanderen' (a province).


#### Participation Columns — Dtype Inconsistency

In [40]:
part_cols = [c for c in clubs_df.columns if c.startswith("Participation")]
print("Participation column dtypes:\n")
for c in part_cols:
    print(f"  {c:<35s} {clubs_df[c].dtype}")

print("\n→ 2016–2019 are bool (True/False), while 2015 and 2022–2025 are float64 (1.0/NaN).")
print("  This must be standardized during ETL.")

Participation column dtypes:

  Participation Games 2015            float64
  Participation Games 2016            bool
  Participation Games 2017            bool
  Participation Games 2018            bool
  Participation Games 2019            bool
  Participation Games 2022            float64
  Participation Games 2023            float64
  Participation Games 2024            float64
  Participation Games 2025            float64

→ 2016–2019 are bool (True/False), while 2015 and 2022–2025 are float64 (1.0/NaN).
  This must be standardized during ETL.


In [41]:
# Show sample values for each type
print("\nSample values (first 5 non-null):\n")
for c in part_cols:
    vals = clubs_df[c].dropna().head(5).tolist()
    print(f"  {c.split()[-1]:>4}: {vals}")


Sample values (first 5 non-null):

  2015: [1.0, 1.0, 1.0, 1.0, 1.0]
  2016: [True, True, True, True, True]
  2017: [True, True, True, True, True]
  2018: [True, False, False, True, True]
  2019: [True, False, True, True, True]
  2022: [1.0, 1.0, 1.0, 1.0, 1.0]
  2023: [1.0, 1.0, 1.0, 1.0, 1.0]
  2024: [1.0, 1.0, 1.0, 1.0, 1.0]
  2025: [1.0, 1.0, 1.0, 1.0, 1.0]


### Missing Value Summary

In [42]:
total_clubs = len(clubs_df)
print("Missing values per column:\n")
print(f"  {'Column':<40} {'Missing':>7} {'%':>7}")
print(f"  {'-'*40} {'-'*7} {'-'*7}")
for col in clubs_df.columns:
    cnt = clubs_df[col].isna().sum()
    print(f"  {col:<40} {cnt:>7} {cnt/total_clubs:>7.1%}")

Missing values per column:

  Column                                   Missing       %
  ---------------------------------------- ------- -------
  Group number                                   0    0.0%
  Name                                           0    0.0%
  Primary language                               0    0.0%
  Address (Street and Number)                    2    0.5%
  Zipcode                                        3    0.7%
  City                                           0    0.0%
  Province                                       6    1.4%
  Country                                       74   17.0%
  Participation Games 2015                     157   36.0%
  Participation Games 2016                       0    0.0%
  Participation Games 2017                       0    0.0%
  Participation Games 2018                       0    0.0%
  Participation Games 2019                       0    0.0%
  Participation Games 2022                     174   39.9%
  Participation Games 2023  

### Key Observations

1. **Group number is a reliable PK** — 436 unique, no nulls.
2. **Name is fully unique** — safe as join key, but fuzzy matching needed.
3. **Country needs major cleanup** — 14 Belgium spelling variants.
4. **Province/Country swap on row 433** (Group 786).
5. **Participation columns have inconsistent dtypes** (bool vs float).
6. **City casing is mixed** — ~63% ALL CAPS.
7. **All clubs are Belgian** except 1 Luxembourg entry.
8. **Zipcode stored as float** — fix the 29900 outlier.

---
## 3. Results Files
(`Thomas More Results YYYY.xlsx`)

### Overview — Load All 9 Files

In [43]:
YEARS = DataLoader.available_years()
results = {}
for year in YEARS:
    results[year] = loader.load_results(year)
    print(f"Loaded {year}: {results[year].shape[0]:,} rows")

print(f"\nTotal records: {sum(df.shape[0] for df in results.values()):,}")

Loaded 2015: 14,756 rows


Loaded 2016: 12,849 rows


Loaded 2017: 14,363 rows


Loaded 2018: 13,554 rows


Loaded 2019: 14,009 rows


Loaded 2022: 9,409 rows


Loaded 2023: 11,798 rows


Loaded 2024: 11,239 rows


Loaded 2025: 10,398 rows

Total records: 112,375


### Per-Year Structure

In [44]:
print(f"{'Year':>6} {'Sheets':>8} {'Cols':>5} {'Rows':>8}  Columns")
print(f"{'-'*6} {'-'*8} {'-'*5} {'-'*8}  {'-'*50}")
for year in YEARS:
    df = results[year]
    print(f"{year:>6} {'Sheet1':>8} {df.shape[1]:>5} {df.shape[0]:>8,}  "
          f"{list(df.columns)}")

  Year   Sheets  Cols     Rows  Columns
------ -------- ----- --------  --------------------------------------------------
  2015   Sheet1    12   14,756  ['Code', 'Club', 'Sport', 'Role', 'DOB', 'Age', 'Gender', 'Event', 'Place', 'Score', 'Summary (all)', 'Year']
  2016   Sheet1    12   12,849  ['Code', 'Club', 'Sport', 'Role', 'DOB', 'Age', 'Gender', 'Event', 'Place', 'Score', 'Summary (all)', 'Year']
  2017   Sheet1    12   14,363  ['Code', 'Club', 'Sport', 'Role', 'DOB', 'Age', 'Gender', 'Event', 'Place', 'Score', 'Summary (all)', 'Year']
  2018   Sheet1    12   13,554  ['Code', 'Club', 'Sport', 'Role', 'DOB', 'Age', 'Gender', 'Event', 'Place', 'Score', 'Summary (all)', 'Year']
  2019   Sheet1    12   14,009  ['Code', 'Club', 'Sport', 'Role', 'DOB', 'Age', 'Gender', 'Event', 'Place', 'Score', 'Summary (all)', 'Year']
  2022   Sheet1    12    9,409  ['Code', 'Club', 'Sport', 'Role', 'DOB', 'Age', 'Gender', 'Event', 'Place', 'Score', 'Summary (all)', 'Year']
  2023   Sheet1    11   1

In [45]:
# Data types per year
print("Age dtype and DOB dtype per year:\n")
for year in YEARS:
    df = results[year]
    print(f"  {year}: Age={df['Age'].dtype}, DOB={df['DOB'].dtype}")

Age dtype and DOB dtype per year:

  2015: Age=int64, DOB=datetime64[ns]
  2016: Age=float64, DOB=datetime64[ns]
  2017: Age=int64, DOB=datetime64[ns]
  2018: Age=int64, DOB=datetime64[ns]
  2019: Age=int64, DOB=datetime64[ns]
  2022: Age=float64, DOB=datetime64[ns]
  2023: Age=float64, DOB=datetime64[ns]
  2024: Age=int64, DOB=datetime64[ns]
  2025: Age=float64, DOB=datetime64[ns]


### Schema Comparison

In [46]:
# Schema comparison using DataProfiler
result_files = [f"Thomas More Results {y}.xlsx" for y in YEARS]
schema_cmp = profiler.compare_schemas(result_files)

print(f"Common columns across all years ({len(schema_cmp['common'])}):")
for col in schema_cmp["common"]:
    print(f"  \u2022 {col}")

print(f"\nPer-file differences:")
for fname, diff in schema_cmp["per_file"].items():
    if diff["added"] or diff["removed"]:
        year = DataLoader.extract_year(fname)
        if diff["added"]:
            print(f"  {year}: extra columns = {diff['added']}")
        if diff["removed"]:
            print(f"  {year}: missing columns = {diff['removed']}")

Common columns across all years (10):
  • Age
  • Club
  • Code
  • DOB
  • Event
  • Gender
  • Place
  • Role
  • Score
  • Sport

Per-file differences:
  2015: extra columns = ['Summary (all)']
  2016: extra columns = ['Summary (all)']
  2017: extra columns = ['Summary (all)']
  2018: extra columns = ['Summary (all)']
  2019: extra columns = ['Summary (all)']
  2022: extra columns = ['Summary (all)']
  2024: extra columns = ['Summary (all)']
  2025: extra columns = ['Summary (all)']


### Column Analysis

#### Code (Athlete ID)

In [47]:
print(f"{'Year':>6} {'Unique Codes':>14} {'Missing':>8}")
print(f"{'-'*6} {'-'*14} {'-'*8}")
for year in YEARS:
    df = results[year]
    unique = df["Code"].dropna().nunique()
    missing = df["Code"].isna().sum()
    print(f"{year:>6} {unique:>14,} {missing:>8}")

  Year   Unique Codes  Missing
------ -------------- --------
  2015          3,288        0
  2016          3,231        4
  2017          3,446        0
  2018          3,419        0
  2019          3,482        0
  2022          2,393       11
  2023          2,969       63
  2024          2,858        0
  2025          2,788        1


In [48]:
# Unique athletes across all years
all_codes = pd.concat(
    [results[y]["Code"].dropna() for y in YEARS], ignore_index=True
)
print(f"Total unique athletes across all years: {all_codes.nunique():,}")

Total unique athletes across all years: 7,607


In [49]:
# Multi-year participation
code_year_presence = {}
for year in YEARS:
    codes_in_year = set(results[year]["Code"].dropna().unique())
    for c in codes_in_year:
        code_year_presence.setdefault(c, set()).add(year)

year_counts = Counter(len(v) for v in code_year_presence.values())
print("Multi-year participation:\n")
print(f"  {'Appeared in N years':>22} {'Athletes':>10}")
print(f"  {'-'*22} {'-'*10}")
for n_years in range(1, len(YEARS) + 1):
    label = f"{n_years} year{'s' if n_years > 1 else ' only'}"
    count = year_counts.get(n_years, 0)
    print(f"  {label:>22} {count:>10,}")

returning = sum(cnt for n, cnt in year_counts.items() if n >= 2)
total_athletes = len(code_year_presence)
print(f"\n  Total unique athletes: {total_athletes:,}")
print(f"  Returning (2+ years):  {returning:,} ({returning/total_athletes:.1%})")
all_nine = year_counts.get(len(YEARS), 0)
print(f"  Competed every year:   {all_nine:,}")

Multi-year participation:

     Appeared in N years   Athletes
  ---------------------- ----------
             1 year only      2,023
                 2 years      1,320
                 3 years        892
                 4 years        847
                 5 years        776
                 6 years        423
                 7 years        396
                 8 years        413
                 9 years        517

  Total unique athletes: 7,607
  Returning (2+ years):  5,584 (73.4%)
  Competed every year:   517


#### Club

In [50]:
print(f"{'Year':>6} {'Distinct Clubs':>16} {'Missing':>8}")
print(f"{'-'*6} {'-'*16} {'-'*8}")
for year in YEARS:
    df = results[year]
    distinct = df["Club"].dropna().nunique()
    missing = df["Club"].isna().sum()
    print(f"{year:>6} {distinct:>16,} {missing:>8}")

  Year   Distinct Clubs  Missing
------ ---------------- --------
  2015              300        0
  2016              304        4
  2017              319        0
  2018              314        0
  2019              319        0
  2022              252       11
  2023              289       63
  2024              285        0
  2025              284        1


In [51]:
# Total distinct raw club names
all_club_names = pd.concat(
    [results[y]["Club"].dropna() for y in YEARS], ignore_index=True
)
print(f"Total distinct club names (raw): {all_club_names.nunique()}")

Total distinct club names (raw): 497


In [52]:
# Spelling variation examples
print("\nKnown spelling variations:\n")
variation_examples = [
    ("BLIJDORP  BUGGENHOUT", "BLIJDORP BUGGENHOUT", "extra space"),
    ("Foyer du Calydon", "Foyer du Calydron", "typo"),
    ("SCHOOL @ WATERKANT", "SCHOOL@WATERKANT", "space around @"),
    ("G - Omnisportclub Vlaamse Ardennen",
     "G-OMNISPORTCLUB VLAAMSE ARDENNEN", "casing + spaces"),
    ("G - SEPPE VZW", "G-SEPPE VZW", "space around hyphen"),
]

for a, b, issue in variation_examples:
    a_found = a in all_club_names.values
    b_found = b in all_club_names.values
    print(f"  '{a}' vs '{b}'  ({issue})  "
          f"[found: {a_found}/{b_found}]")


Known spelling variations:

  'BLIJDORP  BUGGENHOUT' vs 'BLIJDORP BUGGENHOUT'  (extra space)  [found: True/True]
  'Foyer du Calydon' vs 'Foyer du Calydron'  (typo)  [found: True/True]
  'SCHOOL @ WATERKANT' vs 'SCHOOL@WATERKANT'  (space around @)  [found: True/True]
  'G - Omnisportclub Vlaamse Ardennen' vs 'G-OMNISPORTCLUB VLAAMSE ARDENNEN'  (casing + spaces)  [found: True/True]
  'G - SEPPE VZW' vs 'G-SEPPE VZW'  (space around hyphen)  [found: True/True]


In [53]:
# LES QUATRE(S) SAISONS variants
quatre = all_club_names[
    all_club_names.str.contains("QUATRE", case=False, na=False)
].unique()
print("LES QUATRE SAISONS variants:")
for q in sorted(quatre):
    print(f"  • {q}")

LES QUATRE SAISONS variants:
  • LES QUATRE SAISONS
  • LES QUATRE SAISONS 2
  • LES QUATRES SAISONS
  • LES QUATRES SAISONS 2


#### Sport

In [54]:
print(f"{'Year':>6} {'Distinct Sports':>16} {'Missing':>8}")
print(f"{'-'*6} {'-'*16} {'-'*8}")
for year in YEARS:
    df = results[year]
    distinct = df["Sport"].dropna().nunique()
    missing = df["Sport"].isna().sum()
    print(f"{year:>6} {distinct:>16} {missing:>8}")

  Year  Distinct Sports  Missing
------ ---------------- --------
  2015               20        0
  2016               20        0
  2017               20        0
  2018               20        0
  2019               21        0
  2022               16        0
  2023               21        0
  2024               19        0
  2025               18        0


In [55]:
# All distinct sports across all years
all_sports = set()
for year in YEARS:
    all_sports.update(results[year]["Sport"].dropna().unique())

print(f"All {len(all_sports)} distinct sports across all years:\n")
for s in sorted(all_sports):
    print(f"  • {s}")

All 23 distinct sports across all years:

  • Adapted Physical Activities
  • Aquatics/Swimming
  • Athletics/Track and Field
  • Badminton
  • Basketball
  • Bocce
  • Bowling
  • Cycling
  • Equestrian
  • Floorball
  • Football/Soccer
  • Gymnastics (Artistic)
  • Gymnastics (Rhythmic)
  • Judo
  • Kayaking
  • Motor Activities
  • Netball
  • Sailing
  • Sportgames
  • Swimming
  • Table Tennis
  • Tennis
  • Triathlon


In [56]:
# Aquatics/Swimming vs Swimming distinction
print("\nAquatics/Swimming vs Swimming:\n")
for year in YEARS:
    df = results[year]
    sports = df["Sport"].dropna().unique()
    has_aq = "Aquatics/Swimming" in sports
    has_sw = "Swimming" in sports
    if has_aq or has_sw:
        print(f"  {year}: Aquatics/Swimming={'✓' if has_aq else '✗'}  "
              f"Swimming={'✓' if has_sw else '✗'}")


Aquatics/Swimming vs Swimming:

  2015: Aquatics/Swimming=✓  Swimming=✗
  2016: Aquatics/Swimming=✓  Swimming=✗
  2017: Aquatics/Swimming=✓  Swimming=✗
  2018: Aquatics/Swimming=✓  Swimming=✗
  2019: Aquatics/Swimming=✓  Swimming=✗
  2022: Aquatics/Swimming=✓  Swimming=✗
  2023: Aquatics/Swimming=✓  Swimming=✗
  2024: Aquatics/Swimming=✗  Swimming=✓
  2025: Aquatics/Swimming=✗  Swimming=✓


In [57]:
# Event prefixes for each swimming sport
print("\nSample events for Aquatics/Swimming vs Swimming:\n")
for sport_name in ["Aquatics/Swimming", "Swimming"]:
    print(f"  {sport_name}:")
    events = set()
    for year in YEARS:
        mask = results[year]["Sport"] == sport_name
        events.update(results[year].loc[mask, "Event"].dropna().unique())
    for e in sorted(events)[:5]:
        print(f"    • {e}")
    print()


Sample events for Aquatics/Swimming vs Swimming:

  Aquatics/Swimming:
    • AQ 4X25 M Unified Freestyle Relay
    • AQ14 - 25 M Brasse/Schoolslag
    • AQ15 - 50 M Brasse/Schoolslag
    • AQ16 - 100 M Brasse/Schoolslag
    • AQ24 - 25 M Dos/Rugslag

  Swimming:
    • AQ14 - 25 M Brasse/Schoolslag
    • AQ15 - 50 M Brasse/Schoolslag
    • AQ16 - 100 M Brasse/Schoolslag
    • AQ24 - 25 M Dos/Rugslag
    • AQ25 - 50 M Dos/Rugslag



#### Event

In [58]:
print(f"{'Year':>6} {'Distinct Events':>16} {'Missing':>8}")
print(f"{'-'*6} {'-'*16} {'-'*8}")
for year in YEARS:
    df = results[year]
    distinct = df["Event"].dropna().nunique()
    missing = df["Event"].isna().sum()
    print(f"{year:>6} {distinct:>16} {missing:>8}")

  Year  Distinct Events  Missing
------ ---------------- --------
  2015              136        0
  2016              148        0
  2017              140        0
  2018              136        0
  2019              129        0
  2022              118        0
  2023              143        0
  2024              140        0
  2025              138        0


In [59]:
all_events = set()
for year in YEARS:
    all_events.update(results[year]["Event"].dropna().unique())
print(f"Total distinct event names: {len(all_events)}")

Total distinct event names: 377


In [60]:
# Bilingual naming examples
print("\nBilingual naming inconsistencies (examples):\n")

bilingual_examples = [
    "AT17 - Softbalwerpen",
    "AT17 - Lancement",
    "AT16 - Saut en longueur",
    "AT16 - Verspringen",
]
for prefix in bilingual_examples:
    matches = sorted(e for e in all_events if e.startswith(prefix))
    for m in matches:
        print(f"  • {m}")


Bilingual naming inconsistencies (examples):

  • AT17 - Softbalwerpen/Lancement softball
  • AT17 - Lancement de softball/Softbalwerpen
  • AT16 - Saut en longueur sans élan/Staande vertesprong
  • AT16 - Verspringen zonder aanloop/Saut en longueur sans élan


In [61]:
# Typos, copy artifacts, formatting issues
print("Other event name issues:\n")
for pattern, label in [
    ("hoogtespsrong", "typo"),
    ("(copy of)", "copy artifact"),
    ("FINALES FO", "finals prefix"),
    ("Finales  FO", "finals prefix (extra space)"),
]:
    matches = sorted(e for e in all_events if pattern in e)
    for m in matches:
        print(f"  [{label}] {m}")

Other event name issues:

  [typo] APA - Staande hoogtespsrong/Saut en hauteur sans élan
  [copy artifact] (copy of) CY-Divisioning3-15 km
  [copy artifact] (copy of) Divisioning 500m-1Km
  [copy artifact] (copy of) Divisioning- 500m & 1Km Tricycle
  [finals prefix] FINALES FO - 7 A Side Team
  [finals prefix (extra space)] Finales  FO - 7 A Side Team


#### Place (Rank)

In [62]:
print(f"{'Year':>6} {'Distinct':>9} {'Missing':>8} {'Missing %':>10}")
print(f"{'-'*6} {'-'*9} {'-'*8} {'-'*10}")
for year in YEARS:
    df = results[year]
    distinct = df["Place"].dropna().nunique()
    missing = df["Place"].isna().sum()
    pct = missing / len(df)
    print(f"{year:>6} {distinct:>9} {missing:>8,} {pct:>10.1%}")

  Year  Distinct  Missing  Missing %
------ --------- -------- ----------
  2015        19    4,234      28.7%
  2016        24    2,880      22.4%
  2017        34    2,867      20.0%
  2018        37    2,766      20.4%
  2019        36    3,033      21.7%
  2022        31    2,145      22.8%
  2023        37    2,549      21.6%
  2024        33    2,712      24.1%
  2025        27    4,244      40.8%


In [63]:
# All distinct Place values across all years
all_places = set()
for year in YEARS:
    all_places.update(results[year]["Place"].dropna().unique())

print(f"Total distinct Place values: {len(all_places)}\n")

# Categorize
ordinals = sorted(p for p in all_places if p.endswith(("st", "nd", "rd", "th")))
dq_vals = sorted(p for p in all_places if str(p).startswith("DQ"))
dns_vals = sorted(p for p in all_places if str(p).startswith("DN"))
aq_vals = sorted(p for p in all_places if str(p).startswith("AQ"))
other = sorted(all_places - set(ordinals) - set(dq_vals) - set(dns_vals) - set(aq_vals))

print(f"Ordinal positions ({len(ordinals)}):")
print(f"  {ordinals}\n")

print(f"Disqualifications ({len(dq_vals)}):")
for v in dq_vals:
    print(f"  • {v}")

print(f"\nDid Not Start/Finish ({len(dns_vals)}):")
for v in dns_vals:
    print(f"  • {v}")

print(f"\nAquatics levels ({len(aq_vals)}):")
print(f"  {aq_vals[:10]}{'...' if len(aq_vals) > 10 else ''}")
print(f"  (total: {len(aq_vals)} values)")

print(f"\nOther ({len(other)}):")
for v in other:
    print(f"  • {v}")

Total distinct Place values: 63



Ordinal positions (9):
  ['10th', '1st', '2nd', '3rd', '4th', '5th', '6th', '7th', '8th']

Disqualifications (12):
  • DQ
  • DQ-FAL
  • DQ-FOU
  • DQ-GATE
  • DQ-HE
  • DQ-INT
  • DQ-LI
  • DQ-ME
  • DQ-SUIT
  • DQ-TEC
  • DQ-USC
  • DQ: HE

Did Not Start/Finish (4):
  • DNC
  • DNF
  • DNS
  • DNT

Aquatics levels (35):
  ['AQ 0.1', 'AQ 0.10', 'AQ 0.11', 'AQ 0.12', 'AQ 0.13', 'AQ 0.2', 'AQ 0.3', 'AQ 0.4', 'AQ 0.5', 'AQ 0.7']...
  (total: 35 values)

Other (3):
  • Part
  • SW7.2.2
  • ZERO


In [64]:
# DQ inconsistency: DQ-HE vs DQ: HE
print("DQ format inconsistency:")
for v in sorted(dq_vals):
    if "HE" in v:
        print(f"  • '{v}'")

DQ format inconsistency:
  • 'DQ-HE'
  • 'DQ: HE'


#### Score

In [65]:
print(f"{'Year':>6} {'Non-null':>9} {'Total':>8} {'Missing':>8} {'Missing %':>10}")
print(f"{'-'*6} {'-'*9} {'-'*8} {'-'*8} {'-'*10}")
for year in YEARS:
    df = results[year]
    nonnull = df["Score"].notna().sum()
    total_r = len(df)
    miss = df["Score"].isna().sum()
    pct = miss / total_r
    print(f"{year:>6} {nonnull:>9,} {total_r:>8,} {miss:>8,} {pct:>10.1%}")

  Year  Non-null    Total  Missing  Missing %
------ --------- -------- -------- ----------
  2015    10,465   14,756    4,291      29.1%
  2016    10,099   12,849    2,750      21.4%
  2017    11,484   14,363    2,879      20.0%
  2018     9,839   13,554    3,715      27.4%
  2019    10,120   14,009    3,889      27.8%
  2022     7,367    9,409    2,042      21.7%
  2023     8,886   11,798    2,912      24.7%
  2024     8,568   11,239    2,671      23.8%
  2025     6,679   10,398    3,719      35.8%


In [66]:
# Score format pattern examples
print("Score format examples (sampled from all years):\n")

all_scores = pd.concat(
    [results[y]["Score"].dropna() for y in YEARS], ignore_index=True
)

# Time format: X min, Y.ZZ sec
time_scores = all_scores[all_scores.str.contains(r"\d+ min,", na=False)]
print(f"Time format (X min, Y.ZZ sec) — {len(time_scores):,} occurrences:")
for s in time_scores.sample(min(5, len(time_scores)), random_state=42).values:
    print(f"  • {s}")

# Hour format: X hr, Y min, Z.ZZ sec
hr_scores = all_scores[all_scores.str.contains(r"\d+ hr,", na=False)]
print(f"\nHour format (X hr, Y min, Z.ZZ sec) — {len(hr_scores):,} occurrences:")
for s in hr_scores.head(3).values:
    print(f"  • {s}")

# Distance format: Xm, Y.ZZcm
dist_scores = all_scores[all_scores.str.contains(r"\d+m,", na=False)]
print(f"\nDistance format (Xm, Y.ZZcm) — {len(dist_scores):,} occurrences:")
for s in dist_scores.sample(min(5, len(dist_scores)), random_state=42).values:
    print(f"  • {s}")

# Points format: X points
pts_scores = all_scores[all_scores.str.contains(r"points", na=False)]
print(f"\nPoints format (X points) — {len(pts_scores):,} occurrences:")
for s in pts_scores.sample(min(5, len(pts_scores)), random_state=42).values:
    print(f"  • {s}")

# DNS in Score column
dns_in_score = all_scores[all_scores.str.strip() == "DNS"]
print(f"\nDNS in Score column: {len(dns_in_score)} occurrences")

Score format examples (sampled from all years):



Time format (X min, Y.ZZ sec) — 26,449 occurrences:
  • 0 min, 52.48 sec
  • 1 min, 34.56 sec
  • 0 min, 21.06 sec
  • 0 min, 36.70 sec
  • 0 min, 57.12 sec



Hour format (X hr, Y min, Z.ZZ sec) — 105 occurrences:
  • 2 hr, 3 min, 51.93 sec
  • 2 hr, 1 min, 31.17 sec
  • 1 hr, 58 min, 20.10 sec



Distance format (Xm, Y.ZZcm) — 9,038 occurrences:
  • 0m, 56.00cm
  • 0m, 68.00cm
  • 3m, 57.00cm
  • 3m, 29.00cm
  • 1m, 20.00cm



Points format (X points) — 45,097 occurrences:
  • 3.00 points
  • 2665.00 points
  • 8 points
  • 10 points
  • 9 points

DNS in Score column: 242 occurrences


In [67]:
# Pattern coverage check
categorized = len(time_scores) + len(hr_scores) + len(dist_scores) + len(pts_scores) + len(dns_in_score)
uncategorized = len(all_scores) - categorized
print(f"Categorized scores: {categorized:,} / {len(all_scores):,}")
print(f"Uncategorized:      {uncategorized:,}")
if uncategorized > 0:
    mask = (
        ~all_scores.str.contains(r"\d+ min,", na=False)
        & ~all_scores.str.contains(r"\d+ hr,", na=False)
        & ~all_scores.str.contains(r"\d+m,", na=False)
        & ~all_scores.str.contains(r"points", na=False)
        & (all_scores.str.strip() != "DNS")
    )
    print("Samples of uncategorized scores:")
    for s in all_scores[mask].head(10).values:
        print(f"  • '{s}'")

Categorized scores: 80,931 / 83,507
Uncategorized:      2,576


Samples of uncategorized scores:
  • '785'
  • '111'
  • '549'
  • '85'
  • '234'
  • '42'
  • '67'
  • '591'
  • '502'
  • '82'


### Row Count Trends

In [68]:
# Text-based bar chart
max_rows = max(df.shape[0] for df in results.values())
bar_width = 40

print(f"{'Year':>6} | {'Rows':>7} | {'Athletes':>8} | Bar")
print(f"{'-'*6}-+-{'-'*7}-+-{'-'*8}-+-{'-'*42}")
for year in YEARS:
    if year == 2022:
        print(f"{'':>6} | {'':>7} | {'':>8} | --- COVID GAP: 2020, 2021 ---")
    df = results[year]
    n_rows = df.shape[0]
    n_athletes = df["Code"].dropna().nunique()
    bar_len = int(n_rows / max_rows * bar_width)
    bar = "█" * bar_len
    print(f"{year:>6} | {n_rows:>7,} | {n_athletes:>8,} | {bar}")

print(f"\nPre-COVID avg rows:    "
      f"{sum(results[y].shape[0] for y in [2015,2016,2017,2018,2019]) / 5:,.0f}")
print(f"Post-COVID avg rows:   "
      f"{sum(results[y].shape[0] for y in [2022,2023,2024,2025]) / 4:,.0f}")

  Year |    Rows | Athletes | Bar
-------+---------+----------+-------------------------------------------
  2015 |  14,756 |    3,288 | ████████████████████████████████████████
  2016 |  12,849 |    3,231 | ██████████████████████████████████
  2017 |  14,363 |    3,446 | ██████████████████████████████████████
  2018 |  13,554 |    3,419 | ████████████████████████████████████
  2019 |  14,009 |    3,482 | █████████████████████████████████████
       |         |          | --- COVID GAP: 2020, 2021 ---
  2022 |   9,409 |    2,393 | █████████████████████████
  2023 |  11,798 |    2,969 | ███████████████████████████████
  2024 |  11,239 |    2,858 | ██████████████████████████████
  2025 |  10,398 |    2,788 | ████████████████████████████

Pre-COVID avg rows:    13,906
Post-COVID avg rows:   10,711


### Data Quality Issues

#### Duplicate Rows (Code + Event + Sport)

In [69]:
print(f"{'Year':>6} {'Duplicates':>12} {'% of rows':>10}")
print(f"{'-'*6} {'-'*12} {'-'*10}")
for year in YEARS:
    df = results[year]
    dup_mask = df.duplicated(subset=["Code", "Event", "Sport"], keep=False)
    dup_count = dup_mask.sum()
    pct = dup_count / len(df)
    print(f"{year:>6} {dup_count:>12,} {pct:>10.1%}")

print("\n→ NOT actual duplication — same athlete+event appears multiple times")
print("  because the export includes multiple rounds (qualifying, preliminary, final).")

  Year   Duplicates  % of rows
------ ------------ ----------
  2015        7,563      51.3%
  2016        5,836      45.4%
  2017        6,938      48.3%
  2018        6,117      45.1%
  2019        6,735      48.1%
  2022        3,964      42.1%
  2023        5,539      46.9%
  2024        5,008      44.6%
  2025        4,101      39.4%

→ NOT actual duplication — same athlete+event appears multiple times
  because the export includes multiple rounds (qualifying, preliminary, final).


In [70]:
# Show an example of multi-round rows
print("\nExample: a single athlete in a single event across rounds:\n")
for year in YEARS:
    df = results[year]
    dups = df[df.duplicated(subset=["Code", "Event", "Sport"], keep=False)]
    if len(dups) > 0:
        sample_group = dups.groupby(["Code", "Event", "Sport"]).filter(
            lambda g: len(g) >= 2
        )
        if len(sample_group) > 0:
            first_combo = sample_group.iloc[0]
            example = df[
                (df["Code"] == first_combo["Code"])
                & (df["Event"] == first_combo["Event"])
                & (df["Sport"] == first_combo["Sport"])
            ]
            print(f"Year {year}:")
            cols = ["Code", "Sport", "Event", "Place", "Score"]
            if "Summary (all)" in example.columns:
                cols.append("Summary (all)")
            print(example[cols].to_string(index=False))
            print()
            break


Example: a single athlete in a single event across rounds:



Year 2015:
            Code   Sport                Event Place            Score                                               Summary (all)
00BKM94J4INLRQAF Cycling Divisioning 500m-1Km   NaN 5 min, 30.00 sec Qual: 02:34.00   22- 29  Bib#:45  Fin: M6, 5 min, 30.00 sec
00BKM94J4INLRQAF Cycling Divisioning 500m-1Km   NaN              NaN Qual: 02:34.00   22- 29  Bib#:45  Fin: M6, 5 min, 30.00 sec



#### Missing Value Patterns per Year

In [71]:
key_cols = ["Code", "Club", "Sport", "Role", "DOB", "Age", "Gender",
            "Event", "Place", "Score"]

print("Missing values per year (key columns):\n")
header = f"{'Column':<10}"
for y in YEARS:
    header += f" {y:>6}"
print(header)
print("-" * len(header))

for col in key_cols:
    row_str = f"{col:<10}"
    for y in YEARS:
        miss = results[y][col].isna().sum()
        row_str += f" {miss:>6}"
    print(row_str)

Missing values per year (key columns):

Column       2015   2016   2017   2018   2019   2022   2023   2024   2025
-------------------------------------------------------------------------


Code            0      4      0      0      0     11     63      0      1
Club            0      4      0      0      0     11     63      0      1
Sport           0      0      0      0      0      0      0      0      0
Role            0      4      0      0      0     11     63      0      1
DOB             5      4     13      0      8     11     68      6     24
Age             0      4      0      0      0     11     63      0      1
Gender          0      4      0      0      0     11     63      0      1
Event           0      0      0      0      0      0      0      0      0
Place        4234   2880   2867   2766   3033   2145   2549   2712   4244
Score        4291   2750   2879   3715   3889   2042   2912   2671   3719


In [72]:
# 2023: correlated missing (Code+Club+Role+Age+Gender all missing together)
df_2023 = results[2023]
code_missing_2023 = df_2023["Code"].isna()
print(f"2023 — rows with missing Code: {code_missing_2023.sum()}")
corr_cols = ["Code", "Club", "Role", "Age", "Gender"]
all_missing = df_2023[corr_cols].isna().all(axis=1)
print(f"2023 — rows where ALL of {corr_cols} are missing: {all_missing.sum()}")
print("→ These are entire athlete records that are blank (correlated missingness).")

2023 — rows with missing Code: 63
2023 — rows where ALL of ['Code', 'Club', 'Role', 'Age', 'Gender'] are missing: 63
→ These are entire athlete records that are blank (correlated missingness).


#### Gender Inconsistencies

In [73]:
print("Gender values per year:\n")
for year in YEARS:
    df = results[year]
    vals = df["Gender"].value_counts(dropna=False)
    parts = []
    for v, c in vals.items():
        label = "NaN" if pd.isna(v) else str(v)
        parts.append(f"{label}={c}")
    print(f"  {year}: {', '.join(parts)}")

print("\n→ Most years: Male/Female only.")
print("  2016, 2018, 2025: include 'Unknown'.")
print("  2016, 2022, 2023, 2025: have NaN gender values.")
print("  Gender is consistently Male/Female (no M/F abbreviations in Results).")

Gender values per year:

  2015: Male=10236, Female=4520
  2016: Male=8698, Female=4143, Unknown=4, NaN=4
  2017: Male=10044, Female=4319
  2018: Male=9329, Female=4221, Unknown=4
  2019: Male=9591, Female=4418
  2022: Male=6340, Female=3058, NaN=11
  2023: Male=7933, Female=3802, NaN=63
  2024: Male=7629, Female=3610
  2025: Male=6874, Female=3520, Unknown=3, NaN=1

→ Most years: Male/Female only.
  2016, 2018, 2025: include 'Unknown'.
  2016, 2022, 2023, 2025: have NaN gender values.
  Gender is consistently Male/Female (no M/F abbreviations in Results).


### Key Observations

1. **Multi-round data structure:** ~45% of rows are "duplicates" on
   Code+Event+Sport — multiple rounds per athlete/event. ETL must
   implement round disambiguation logic.
2. **Score parsing is simpler than expected:** Consistent verbose format
   (`X min, Y.ZZ sec` / `Xm, Y.ZZcm` / `X.XX points`).
3. **Post-COVID decline:** ~25% participation drop after 2020–2021 gap.
4. **2023 most problematic:** Missing `Summary (all)`, highest Code
   missing rate (63 rows).
5. **2025 highest missing rates:** Place 40.8%, Score 35.8%.
6. **Event normalization critical:** 375 distinct names, many bilingual flips.
7. **Club fuzzy matching needed:** 491 raw names with variations.
8. **Aquatics dual naming:** `Aquatics/Swimming` (AQ) and `Swimming` (SW)
   are separate — do NOT merge.

## 4. Cross-File Relationships

In [74]:
# Section 4: Cross-file analysis
# Data already loaded via DataLoader — just prepare combined results
cert_df = loader.load_certifications()
cert_df = cert_df.dropna(how="all")  # Drop 780 empty rows
clubs_df = loader.load_clubs()
results_df = loader.load_all_results()

print(f"Certifications: {len(cert_df):,} rows (after dropping empties)")
print(f"Clubs: {len(clubs_df):,} rows")
print(f"Results (all years): {len(results_df):,} rows")

Certifications: 20,221 rows (after dropping empties)
Clubs: 436 rows
Results (all years): 112,375 rows


### 4.1 Code Linkage (Results ↔ Certifications)

The `Code` column is the primary join key between Results and the Certifications
master list. We check how many athlete codes in Results have a corresponding
Certifications record.

In [75]:
# Overall match rate
result_codes = results_df["Code"].dropna().unique()
cert_codes = cert_df["Code"].dropna().unique()

matched = set(result_codes) & set(cert_codes)
orphan_in_results = set(result_codes) - set(cert_codes)
cert_only = set(cert_codes) - set(result_codes)

summary = pd.DataFrame(
    {
        "Metric": [
            "Codes in Results",
            "Codes in Certifications",
            "Matched",
            "Orphan in Results (not in Cert)",
            "Cert-only (never competed)",
        ],
        "Count": [
            len(result_codes),
            len(cert_codes),
            len(matched),
            len(orphan_in_results),
            len(cert_only),
        ],
    }
)
summary["% of Results"] = summary["Count"].apply(
    lambda x: f"{x / len(result_codes) * 100:.1f}%" if x <= len(result_codes) else "—"
)
# Override for cert-only (% of Certifications)
summary.loc[summary["Metric"] == "Cert-only (never competed)", "% of Results"] = (
    f"{len(cert_only) / len(cert_codes) * 100:.1f}% of Cert"
)
summary.loc[summary["Metric"] == "Codes in Certifications", "% of Results"] = "—"
print(summary.to_string(index=False))# Cross-check with DataProfiler
integrity = DataProfiler.check_referential_integrity(
    results_df, cert_df, "Code", "Code"
)
print(f"\nDataProfiler integrity check:")
print(f"  Orphans in Results (not in Cert): {integrity['left_only_count']}")
print(f"  Orphans in Cert (not in Results): {integrity['right_only_count']}")


                         Metric  Count  % of Results
               Codes in Results   7607        100.0%
        Codes in Certifications  20221             —
                        Matched   7579         99.6%
Orphan in Results (not in Cert)     28          0.4%
     Cert-only (never competed)  12642 62.5% of Cert



DataProfiler integrity check:


  Orphans in Results (not in Cert): 28
  Orphans in Cert (not in Results): 12642


In [76]:
# Per-year breakdown
per_year = []
for year, grp in results_df.groupby("Year"):
    year_codes = grp["Code"].dropna().unique()
    n_matched = len(set(year_codes) & set(cert_codes))
    n_orphans = len(set(year_codes) - set(cert_codes))
    per_year.append(
        {
            "Year": year,
            "Unique Codes": len(year_codes),
            "Matched": n_matched,
            "Match %": f"{n_matched / len(year_codes) * 100:.1f}%",
            "Orphans": n_orphans,
            "Orphan %": f"{n_orphans / len(year_codes) * 100:.1f}%",
        }
    )

per_year_df = pd.DataFrame(per_year)
print(per_year_df.to_string(index=False))

 Year  Unique Codes  Matched Match %  Orphans Orphan %
 2015          3288     3287  100.0%        1     0.0%
 2016          3231     3230  100.0%        1     0.0%
 2017          3446     3445  100.0%        1     0.0%
 2018          3419     3417   99.9%        2     0.1%
 2019          3482     3469   99.6%       13     0.4%
 2022          2393     2391   99.9%        2     0.1%
 2023          2969     2946   99.2%       23     0.8%
 2024          2858     2856   99.9%        2     0.1%
 2025          2788     2788  100.0%        0     0.0%


In [77]:
# Person types of matched codes
matched_cert = cert_df[cert_df["Code"].isin(matched)]
person_type_counts = matched_cert["Person type"].value_counts()
print("Person types of matched records (Results ↔ Certifications):")
print(person_type_counts.to_string())
print(f"\nTotal matched: {len(matched):,}")

Person types of matched records (Results ↔ Certifications):
Person type
Athlete            7347
Unified Partner     169
Coach                63

Total matched: 7,579


In [78]:
# Cert-only records breakdown by Person type
cert_only_df = cert_df[cert_df["Code"].isin(cert_only)]
print(f"Cert-only records (never competed): {len(cert_only_df):,}")
print(f"  = {len(cert_only) / len(cert_codes) * 100:.1f}% of all Certifications\n")
print("Person types of cert-only records:")
print(cert_only_df["Person type"].value_counts().to_string())

Cert-only records (never competed): 12,642
  = 62.5% of all Certifications

Person types of cert-only records:


Person type
Athlete            8517
Coach              3738
Unified Partner     347
Staff                19
Volunteer             6
Head Coach            3
VIP                   2
Medical               2
Security              2
A-HOD                 2
Manager               1
Family member         1
Media                 1
AS-Staff              1

In [79]:
# Orphan codes analysis — codes in Results not found in Certifications
orphan_codes = sorted(orphan_in_results)
print(f"Orphan codes in Results (not in Certifications): {len(orphan_codes)}")

# Which years do orphans appear in?
orphan_rows = results_df[results_df["Code"].isin(orphan_codes)]
print(f"\nTotal result rows for orphan codes: {len(orphan_rows)}")
print("\nOrphan codes per year:")
print(orphan_rows.groupby("Year")["Code"].nunique().to_string())
print(f"\nSample orphan codes: {orphan_codes[:10]}")

Orphan codes in Results (not in Certifications): 28

Total result rows for orphan codes: 119

Orphan codes per year:
Year
2015     1
2016     1
2017     1
2018     2
2019    13
2022     2
2023    23
2024     2

Sample orphan codes: ['06J63B8U7KIRSR1J', '3A54EWIN33J0MOKX', '3F5N30BLJ81JC33F', '7LFVV2H4Y7CV4EL6', '84NFFDORH484MIHH', '85J1YJ7LPYIQH9OW', '8SKTBLEOSH590PQL', 'AO4LZBAO3PG1WERQ', 'ASC4BWQHN7PUBJHZ', 'B4FPCNAAVDIYZQT7']


### 4.2 Club Linkage (Results ↔ Clubs)

The `Club` column in Results must be matched to `Name` in the Clubs master file.
Unlike Code, this is a **name-based join** with no shared numeric key.

In [80]:
# Exact match rate
result_club_names = results_df["Club"].dropna().unique()
clubs_names = clubs_df["Name"].dropna().unique()

exact_match = set(result_club_names) & set(clubs_names)
no_match = set(result_club_names) - set(clubs_names)

print("Club name matching (Results ↔ Clubs):")
print(f"  Unique club names in Results: {len(result_club_names)}")
print(f"  Unique club names in Clubs:   {len(clubs_names)}")
print(f"  Exact match:                  {len(exact_match)} ({len(exact_match)/len(result_club_names)*100:.1f}%)")
print(f"  No match:                     {len(no_match)} ({len(no_match)/len(result_club_names)*100:.1f}%)")

Club name matching (Results ↔ Clubs):
  Unique club names in Results: 497
  Unique club names in Clubs:   436
  Exact match:                  375 (75.5%)
  No match:                     122 (24.5%)


In [81]:
# Case-insensitive + whitespace-stripped matching
result_clubs_norm = {c: c.strip().lower() for c in result_club_names}
clubs_norm = {c: c.strip().lower() for c in clubs_names}

norm_clubs_set = set(clubs_norm.values())
ci_match = {c for c, n in result_clubs_norm.items() if n in norm_clubs_set}
ci_no_match = set(result_club_names) - ci_match

print(f"\nAfter case-insensitive + whitespace normalization:")
print(f"  Matched clubs:   {len(ci_match)} (was {len(exact_match)} exact)")
print(f"  Unmatched clubs:  {len(ci_no_match)} (was {len(no_match)})")
print(f"  Improvement:      +{len(ci_match) - len(exact_match)} clubs recovered")


After case-insensitive + whitespace normalization:
  Matched clubs:   385 (was 375 exact)
  Unmatched clubs:  112 (was 122)
  Improvement:      +10 clubs recovered


In [82]:
# Row-level coverage
exact_match_rows = results_df[results_df["Club"].isin(exact_match)]
ci_match_rows = results_df[results_df["Club"].isin(ci_match)]
total_rows = len(results_df)

coverage = pd.DataFrame(
    {
        "Match Level": [
            "Exact match",
            "Case-insensitive + whitespace-stripped",
            "Unmatched after normalization",
        ],
        "Rows Matched": [
            len(exact_match_rows),
            len(ci_match_rows),
            total_rows - len(ci_match_rows),
        ],
        "Total Rows": [total_rows] * 3,
        "Coverage": [
            f"{len(exact_match_rows)/total_rows*100:.1f}%",
            f"{len(ci_match_rows)/total_rows*100:.1f}%",
            f"{(total_rows - len(ci_match_rows))/total_rows*100:.1f}%",
        ],
    }
)
print(coverage.to_string(index=False))

                           Match Level  Rows Matched  Total Rows Coverage
                           Exact match         95132      112375    84.7%
Case-insensitive + whitespace-stripped         95777      112375    85.2%
         Unmatched after normalization         16598      112375    14.8%


In [83]:
# Examples of case-insensitive recoveries
recovered_clubs = ci_match - exact_match
print("Clubs recovered by case-insensitive + whitespace matching:")
for rc in sorted(recovered_clubs):
    norm_val = result_clubs_norm[rc]
    # Find the corresponding Clubs master entry
    master_match = [c for c, n in clubs_norm.items() if n == norm_val]
    for mm in master_match:
        print(f"  Results: {repr(rc):40s}  →  Clubs: {repr(mm)}")

Clubs recovered by case-insensitive + whitespace matching:
  Results: '2Gether On Wheels'                       →  Clubs: '2GETHER ON WHEELS'
  Results: 'Foyer du Calydon'                        →  Clubs: 'FOYER DU CALYDON'
  Results: 'G-BASKET LOMMEL '                        →  Clubs: 'G-BASKET LOMMEL'
  Results: 'HOUTLAND ATLETIEKCLUB VZW G-TEAM ACHILLES '  →  Clubs: 'HOUTLAND ATLETIEKCLUB VZW G-TEAM ACHILLES'
  Results: 'KOG-STASEGEM G-VOETBAL '                 →  Clubs: 'KOG-STASEGEM G-VOETBAL'
  Results: "LOTUS D'OCQUIER "                        →  Clubs: "LOTUS D'OCQUIER"
  Results: 'La Caravelle'                            →  Clubs: 'LA CARAVELLE'
  Results: 'OSTEND TENNIS CLUB '                     →  Clubs: 'OSTEND TENNIS CLUB'
  Results: 'VISÉ - CENTRE DE CERFONTAINE '           →  Clubs: 'VISÉ - CENTRE DE CERFONTAINE'
  Results: 'VZW KOMPAS '                             →  Clubs: 'VZW KOMPAS'


In [84]:
# Top 20 unmatched clubs by result row count
unmatched_rows = results_df[results_df["Club"].isin(ci_no_match)]
top_unmatched = (
    unmatched_rows.groupby("Club")
    .size()
    .reset_index(name="Rows")
    .sort_values("Rows", ascending=False)
    .head(20)
)
print("Top 20 unmatched clubs by result row count:")
print(top_unmatched.to_string(index=False))

Top 20 unmatched clubs by result row count:


                            Club  Rows
                   REINE FABIOLA  2954
                      ZWART GOOR   781
             FUNAMBULES DE WAVRE   590
                      DE MEANDER   562
B.S.C. LEISTUNGSZENTRUM DER D.G.   513
    L'EVEIL CENTRE REINE FABIOLA   509
                  LES CHAFRIPONS   478
                       A.C. LYRA   382
               OOSTHEUVEL-TEVONA   358
                        AT ZODAS   355
                         ROTONDE   345
                      KERCKSTEDE   339
                        BBC GEEL   336
                       DEN DRIES   302
                     SKALA AALST   269
            BLIJDORP  BUGGENHOUT   269
                       FUN GYM B   258
          DAGCENTRUM DENDERMONDE   254
                    ROZENWINGERD   251
                         DE LORK   249


### 4.3 Internal Consistency

#### Certifications.Club vs Clubs.Name

In [85]:
# Check how many unique Certifications Club values match Clubs.Name
cert_club_names = cert_df["Club"].dropna().unique()
cert_clubs_matched = set(cert_club_names) & set(clubs_names)
cert_clubs_unmatched = set(cert_club_names) - set(clubs_names)

print("Certifications.Club vs Clubs.Name:")
print(f"  Unique clubs in Certifications: {len(cert_club_names)}")
print(f"  Matched to Clubs.Name:          {len(cert_clubs_matched)} ({len(cert_clubs_matched)/len(cert_club_names)*100:.1f}%)")
print(f"  Not in Clubs.Name:              {len(cert_clubs_unmatched)} ({len(cert_clubs_unmatched)/len(cert_club_names)*100:.1f}%)")

Certifications.Club vs Clubs.Name:


  Unique clubs in Certifications: 434
  Matched to Clubs.Name:          421 (97.0%)
  Not in Clubs.Name:              13 (3.0%)


In [86]:
# List the 13 unmatched Certifications clubs with member counts
unmatched_members = (
    cert_df[cert_df["Club"].isin(cert_clubs_unmatched)]
    .groupby("Club")
    .size()
    .reset_index(name="Members")
    .sort_values("Members", ascending=False)
)
print("Unmatched Certifications clubs (not in Clubs master):")
print(unmatched_members.to_string(index=False))

Unmatched Certifications clubs (not in Clubs master):
                       Club  Members
                    General     7310
                SKALA AALST       25
DARING CLUB LEUVEN ATLETIEK       20
                KV Mechelen       10
        Levante UD Valencia        8
                       BOKA        6
                 THALEIA OK        6
              L'ARBORESPORT        5
                    'T VEER        5
                 DE PELGRIM        4
               LA SAPINIERE        4
                       KVKG        2
            HAGEN (Germany)        1


#### Athlete Club Consistency (Certifications vs Results)

For athletes appearing in both files, how often does their Certifications club
match the club they competed under in Results?

In [87]:
# Build athlete-club pairs from Results (one row per athlete-club combination)
athlete_result_clubs = (
    results_df[results_df["Code"].isin(matched)]
    .groupby("Code")["Club"]
    .apply(lambda x: set(x.dropna()))
    .reset_index()
)

# Get Certifications club per athlete
cert_club_map = (
    cert_df[cert_df["Code"].isin(matched)][["Code", "Club"]]
    .drop_duplicates()
    .set_index("Code")["Club"]
)

# Count code+club pairs (each athlete may have competed under multiple clubs)
code_club_pairs = (
    results_df[results_df["Code"].isin(matched)][["Code", "Club"]]
    .drop_duplicates()
)
print(f"Total Code+Club pairs: {len(code_club_pairs):,}")

# Exact match: cert club appears among the Result clubs for that athlete
exact_club_match = 0
ci_club_match = 0
for _, row in code_club_pairs.iterrows():
    code = row["Code"]
    result_club = row["Club"]
    cert_club = cert_club_map.get(code, None)
    if cert_club is not None and result_club is not None:
        if cert_club == result_club:
            exact_club_match += 1
        if str(cert_club).strip().lower() == str(result_club).strip().lower():
            ci_club_match += 1

print(f"Exact club match (Cert Club = Result Club): {exact_club_match} ({exact_club_match/len(code_club_pairs)*100:.1f}%)")
print(f"Case-insensitive match:                     {ci_club_match} ({ci_club_match/len(code_club_pairs)*100:.1f}%)")

# Athletes where NO result club matches their cert club
athletes_any_match = set()
athletes_no_match = set()
for code in matched:
    cert_club = cert_club_map.get(code, None)
    if cert_club is None:
        continue
    result_clubs = set(
        results_df.loc[
            (results_df["Code"] == code) & results_df["Club"].notna(), "Club"
        ]
    )
    cert_norm = str(cert_club).strip().lower()
    result_norms = {str(c).strip().lower() for c in result_clubs}
    if cert_norm in result_norms:
        athletes_any_match.add(code)
    else:
        athletes_no_match.add(code)

print(f"\nAthletes with ≥1 matching club year:     {len(athletes_any_match)} ({len(athletes_any_match)/len(matched)*100:.1f}%)")
print(f"Athletes with NO matching club any year:  {len(athletes_no_match)} ({len(athletes_no_match)/len(matched)*100:.1f}%)")

Total Code+Club pairs: 8,896


Exact club match (Cert Club = Result Club): 4753 (53.4%)
Case-insensitive match:                     4805 (54.0%)



Athletes with ≥1 matching club year:     4779 (63.1%)
Athletes with NO matching club any year:  2800 (36.9%)


In [88]:
# Athletes competing under multiple clubs over time
multi_club = (
    results_df[results_df["Code"].isin(matched)]
    .groupby("Code")["Club"]
    .nunique()
)
multi_club_athletes = multi_club[multi_club > 1]
print(f"Athletes competing under multiple clubs: {len(multi_club_athletes)} ({len(multi_club_athletes)/len(matched)*100:.1f}%)")
print(f"\nDistribution of clubs per athlete:")
print(multi_club.value_counts().sort_index().to_string())

Athletes competing under multiple clubs: 1162 (15.3%)

Distribution of clubs per athlete:
Club
1    6417
2    1025
3     121
4      14
5       2


#### Sport Naming Consistency Across Years

In [89]:
# Distinct sports per year
sport_per_year = results_df.groupby("Year")["Sport"].nunique().reset_index()
sport_per_year.columns = ["Year", "Distinct Sports"]
print(sport_per_year.to_string(index=False))

 Year  Distinct Sports
 2015               20
 2016               20
 2017               20
 2018               20
 2019               21
 2022               16
 2023               21
 2024               19
 2025               18


In [90]:
# Which sports appear in all years vs. some years?
years = sorted(results_df["Year"].unique())
sport_year_presence = results_df.groupby("Sport")["Year"].apply(set).reset_index()
sport_year_presence["Years Present"] = sport_year_presence["Year"].apply(len)
sport_year_presence["All Years?"] = sport_year_presence["Years Present"] == len(years)

stable_sports = sport_year_presence[sport_year_presence["All Years?"]].sort_values("Sport")
print(f"\nSports present in ALL {len(years)} years ({len(stable_sports)} sports):")
for s in stable_sports["Sport"]:
    print(f"  {s}")

all_sports = sorted(results_df["Sport"].dropna().unique())
print(f"\nTotal distinct sports across all years: {len(all_sports)}")


Sports present in ALL 9 years (14 sports):
  Adapted Physical Activities
  Athletics/Track and Field
  Badminton
  Basketball
  Bocce
  Floorball
  Gymnastics (Artistic)
  Gymnastics (Rhythmic)
  Motor Activities
  Netball
  Sportgames
  Table Tennis
  Tennis
  Triathlon



Total distinct sports across all years: 23


In [91]:
# Sports with inconsistent presence across years
unstable = sport_year_presence[~sport_year_presence["All Years?"]].copy()
unstable["Present In"] = unstable["Year"].apply(lambda s: sorted(s))
unstable["Absent From"] = unstable["Year"].apply(
    lambda s: sorted(set(years) - s)
)
unstable = unstable.sort_values("Years Present", ascending=False)
print("Sports with INCONSISTENT presence across years:")
for _, row in unstable.iterrows():
    print(f"  {row['Sport']:30s}  Present: {row['Present In']}  |  Absent: {row['Absent From']}")

# Highlight the Aquatics/Swimming rename
aq_years = sport_year_presence.loc[
    sport_year_presence["Sport"] == "Aquatics/Swimming", "Year"
].values
sw_years = sport_year_presence.loc[
    sport_year_presence["Sport"] == "Swimming", "Year"
].values
if len(aq_years) > 0 and len(sw_years) > 0:
    print(f"\n⚠ Aquatics/Swimming present in: {sorted(aq_years[0])}")
    print(f"  Swimming present in:          {sorted(sw_years[0])}")
    print("  → Likely a rename, not two separate sports")

Sports with INCONSISTENT presence across years:
  Bowling                         Present: [2015, 2016, 2017, 2018, 2019, 2023, 2024, 2025]  |  Absent: [np.int64(2022)]
  Equestrian                      Present: [2015, 2016, 2017, 2018, 2019, 2023, 2024, 2025]  |  Absent: [np.int64(2022)]
  Cycling                         Present: [2015, 2016, 2017, 2018, 2019, 2023, 2024, 2025]  |  Absent: [np.int64(2022)]
  Judo                            Present: [2015, 2016, 2017, 2018, 2019, 2022, 2023, 2024]  |  Absent: [np.int64(2025)]
  Aquatics/Swimming               Present: [2015, 2016, 2017, 2018, 2019, 2022, 2023]  |  Absent: [np.int64(2024), np.int64(2025)]
  Football/Soccer                 Present: [2015, 2016, 2017, 2018, 2019]  |  Absent: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Sailing                         Present: [2019, 2023]  |  Absent: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2022), np.int64(2024), np.int64(2025)]
  Swi

### 4.4 Join Key Summary

In [92]:
# Compile the join key summary table
join_summary = pd.DataFrame(
    {
        "Relationship": [
            "Results → Certifications",
            "Results → Clubs",
            "Certifications → Clubs",
            "Cert Club ↔ Results Club",
            "Sport names across years",
        ],
        "Left Key": ["Code", "Club (name)", "Club", "Club (per athlete)", "Sport"],
        "Right Key": ["Code", "Name", "Name", "Club (per athlete)", "Sport"],
        "Match Rate": [
            f"99.6% ({len(matched):,}/{len(result_codes):,})",
            f"{len(exact_match)/len(result_club_names)*100:.1f}% exact / {len(ci_match)/len(result_club_names)*100:.1f}% case-insensitive",
            f"{len(cert_clubs_matched)/len(cert_club_names)*100:.1f}% ({len(cert_clubs_matched)}/{len(cert_club_names)})",
            f"{ci_club_match/len(code_club_pairs)*100:.1f}%",
            f"{len(stable_sports)}/{len(all_sports)} stable",
        ],
        "Action Needed": [
            "Drop 28 orphan codes or create stubs; safe direct join",
            "Fuzzy matching required; ~16.5k rows at risk",
            f"{len(cert_clubs_unmatched)} unmatched; General (7,310 rows) is dominant gap",
            "Club assignment differs; use Results Club per year",
            "Normalize Aquatics/Swimming ↔ Swimming; handle discontinued sports",
        ],
    }
)
print(join_summary.to_string(index=False))

            Relationship           Left Key          Right Key                           Match Rate                                                      Action Needed
Results → Certifications               Code               Code                  99.6% (7,579/7,607)             Drop 28 orphan codes or create stubs; safe direct join
         Results → Clubs        Club (name)               Name 75.5% exact / 77.5% case-insensitive                       Fuzzy matching required; ~16.5k rows at risk
  Certifications → Clubs               Club               Name                      97.0% (421/434)                 13 unmatched; General (7,310 rows) is dominant gap
Cert Club ↔ Results Club Club (per athlete) Club (per athlete)                                54.0%                 Club assignment differs; use Results Club per year
Sport names across years              Sport              Sport                         14/23 stable Normalize Aquatics/Swimming ↔ Swimming; handle discontinued sport

---
## 5. Data Quality Summary

Compilation of all data quality issues found across sections 1–4, ranked by
severity and impact on the ETL pipeline and downstream dashboard use-cases.

### 5.1 Critical Issues (Must Fix for ETL)

Issues that would **break the pipeline or produce incorrect results** if
left unaddressed.

In [93]:
# Quantify critical issues from all previous analyses
print("=" * 80)
print("CRITICAL ISSUES SUMMARY")
print("=" * 80)

# C1: Empty rows in Certifications
cert_raw = pd.read_excel(
    os.path.join(RAW_DIR, "Thomas More Data Certifications.xlsx"), engine="openpyxl"
)
empty_rows = cert_raw.shape[0] - cert_df.shape[0]
print(f"\nC1: Empty rows in Certifications: {empty_rows}")

# C2: Club name matching gap
unmatched_club_rows = total_rows - len(ci_match_rows)
print(f"C2: Club name fuzzy matching gap: {len(ci_no_match)} unmatched clubs → {unmatched_club_rows:,} rows ({unmatched_club_rows/total_rows*100:.1f}%)")

# C3: Score format heterogeneity
score_non_null = results_df["Score"].notna().sum()
print(f"C3: Score values requiring parsing: {score_non_null:,} non-null scores across all years")

# C4: Place/Rank parsing
place_non_null = results_df["Place"].notna().sum()
distinct_places = results_df["Place"].dropna().nunique()
print(f"C4: Place values requiring parsing: {place_non_null:,} non-null with {distinct_places} distinct values")

# C5: General club
general_count = cert_df[cert_df["Club"] == "General"].shape[0]
general_in_results = results_df[
    results_df["Code"].isin(cert_df.loc[cert_df["Club"] == "General", "Code"])
]["Code"].nunique()
print(f"C5: 'General' club members: {general_count:,} in Certifications, {general_in_results:,} appear in Results with real clubs")

# C6: Country spelling variants
country_variants = clubs_df["Country"].dropna().nunique()
print(f"C6: Country column distinct values: {country_variants} (should be ~1 for Belgium)")

# C7: Sport name instability
print(f"C7: Sport name variants: {len(all_sports)} total, {len(stable_sports)} stable across all years")

CRITICAL ISSUES SUMMARY



C1: Empty rows in Certifications: 780
C2: Club name fuzzy matching gap: 112 unmatched clubs → 16,598 rows (14.8%)
C3: Score values requiring parsing: 83,507 non-null scores across all years
C4: Place values requiring parsing: 84,945 non-null with 63 distinct values
C5: 'General' club members: 7,310 in Certifications, 2,267 appear in Results with real clubs
C6: Country column distinct values: 14 (should be ~1 for Belgium)
C7: Sport name variants: 23 total, 14 stable across all years


In [94]:
# DQ code variants in Place column
dq_values = results_df.loc[
    results_df["Place"].astype(str).str.startswith("DQ"), "Place"
].unique()
print(f"DQ code variants found: {len(dq_values)}")
for v in sorted(dq_values):
    count = (results_df["Place"] == v).sum()
    print(f"  {v:15s}  ({count:,} rows)")

# DNS/DNF/DNC/DNT values
non_completion = results_df.loc[
    results_df["Place"].isin(["DNS", "DNF", "DNC", "DNT"]), "Place"
]
print(f"\nNon-completion codes:")
print(non_completion.value_counts().to_string())

DQ code variants found: 12
  DQ               (298 rows)
  DQ-FAL           (1 rows)
  DQ-FOU           (6 rows)
  DQ-GATE          (4 rows)
  DQ-HE            (1,920 rows)
  DQ-INT           (5 rows)


  DQ-LI            (79 rows)
  DQ-ME            (227 rows)
  DQ-SUIT          (3 rows)
  DQ-TEC           (56 rows)
  DQ-USC           (1 rows)
  DQ: HE           (23 rows)

Non-completion codes:
Place
DNS    1127
DNF      36
DNC      19
DNT       5


### 5.2 Medium Issues (Should Fix)

In [95]:
print("=" * 80)
print("MEDIUM ISSUES SUMMARY")
print("=" * 80)

# M1: DOB missing
dob_missing = cert_df["DOB"].isna().sum()
dob_missing_beyond_empty = dob_missing - empty_rows  # beyond empty rows (already dropped)
print(f"\nM1: DOB missing in Certifications: {dob_missing} (all non-empty rows)")
# Breakdown by Person type
dob_missing_by_type = cert_df[cert_df["DOB"].isna()]["Person type"].value_counts()
print(f"    DOB missing by Person type:")
print(dob_missing_by_type.to_string())

# M2: Sentinel DOB (1900-01-02)
sentinel_dob = cert_df[cert_df["DOB"] == pd.Timestamp("1900-01-02")]
print(f"\nM2: Sentinel DOB (1900-01-02): {len(sentinel_dob)} records")

# M3: Age = 0
age_zero = cert_df[cert_df["Age"] == 0]
print(f"M3: Age = 0 records: {len(age_zero)}")

# M4: Province normalization
province_count = clubs_df["Province"].dropna().nunique()
print(f"\nM4: Distinct Province values: {province_count} (should be ~11)")

# M5: City casing
cities = clubs_df["City"].dropna()
all_caps_cities = cities.apply(lambda x: x == x.upper()).sum()
print(f"M5: City casing — ALL CAPS: {all_caps_cities}/{len(cities)} ({all_caps_cities/len(cities)*100:.0f}%)")

# M6: Club name variants in Results
print(f"M6: Raw club name variants in Results: {len(result_club_names)}")

# M7: Bilingual event names
total_events = results_df["Event"].dropna().nunique()
print(f"M7: Distinct event names across all years: {total_events}")

# M8: 2025 missingness
r2025 = results_df[results_df["Year"] == 2025]
place_missing_2025 = r2025["Place"].isna().mean() * 100
score_missing_2025 = r2025["Score"].isna().mean() * 100
print(f"\nM8: 2025 missingness — Place: {place_missing_2025:.1f}%, Score: {score_missing_2025:.1f}%")

# M9: Summary (all) column missing in 2023
for f in result_files:
    year = int(f.split()[-1].replace(".xlsx", ""))
    df = pd.read_excel(os.path.join(RAW_DIR, f), engine="openpyxl", nrows=0)
    has_summary = "Summary (all)" in df.columns
    if not has_summary:
        print(f"M9: 'Summary (all)' column MISSING in {year}")

# M10: Cert club vs Results club consistency
print(f"\nM10: Cert Club ↔ Results Club match: {ci_club_match/len(code_club_pairs)*100:.1f}% of pairs")

MEDIUM ISSUES SUMMARY

M1: DOB missing in Certifications: 1210 (all non-empty rows)
    DOB missing by Person type:
Person type
Coach              754
Athlete            271
Unified Partner    158
Staff               17
Volunteer            5
Manager              1
Security             1
Head Coach           1
A-HOD                1
AS-Staff             1

M2: Sentinel DOB (1900-01-02): 6 records
M3: Age = 0 records: 1210

M4: Distinct Province values: 24 (should be ~11)
M5: City casing — ALL CAPS: 277/436 (64%)
M6: Raw club name variants in Results: 497
M7: Distinct event names across all years: 377

M8: 2025 missingness — Place: 40.8%, Score: 35.8%


M9: 'Summary (all)' column MISSING in 2023



M10: Cert Club ↔ Results Club match: 54.0% of pairs


### 5.3 Low Issues (Nice to Fix)

In [96]:
print("=" * 80)
print("LOW ISSUES SUMMARY")
print("=" * 80)

# L1: Unknown gender
u_gender = cert_df[cert_df["Gender"] == "U"]
print(f"\nL1: 'U' (Unknown) gender values in Certifications: {len(u_gender)}")

# L2: Zipcode outlier
zip_outlier = clubs_df[clubs_df["Zipcode"].astype(str).str.len() > 4]
if len(zip_outlier) > 0:
    print(f"L2: Zipcode outlier(s):")
    print(zip_outlier[["Name", "Zipcode"]].to_string(index=False))

# L3: Person types distribution
print(f"\nL3: Person type distribution in Certifications:")
pt_dist = cert_df["Person type"].value_counts()
pt_dist_pct = (pt_dist / len(cert_df) * 100).round(1)
for pt, cnt in pt_dist.items():
    print(f"    {pt:30s}  {cnt:6,}  ({pt_dist_pct[pt]:.1f}%)")

# L4: Certificate columns sparsity
cert_cols = [
    "Mental Handicap (SOB has this certificate)",
    "Parents Consent (SOB has this certificate)",
    "HAP (SOB has this certificate)",
    "Unified Partner (SOB has this certificate)",
]
print(f"\nL4: Certificate column NaN rates:")
for col in cert_cols:
    pct = cert_df[col].isna().mean() * 100
    print(f"    {col:50s}  {pct:.1f}% NaN")

# L5 & L6: Sport presence notes
print(f"\nL5: Football/Soccer present in: {sorted(sport_year_presence.loc[sport_year_presence['Sport'] == 'Football/Soccer', 'Year'].values[0]) if 'Football/Soccer' in sport_year_presence['Sport'].values else 'N/A'}")
if "Kayaking" in sport_year_presence["Sport"].values:
    print(f"L6: Kayaking present in: {sorted(sport_year_presence.loc[sport_year_presence['Sport'] == 'Kayaking', 'Year'].values[0])}")
if "Sailing" in sport_year_presence["Sport"].values:
    print(f"    Sailing present in: {sorted(sport_year_presence.loc[sport_year_presence['Sport'] == 'Sailing', 'Year'].values[0])}")

LOW ISSUES SUMMARY

L1: 'U' (Unknown) gender values in Certifications: 51
L2: Zipcode outlier(s):
                                     Name  Zipcode
                               LA PILERIE   6590.0
                                      BAM   6600.0
                     LES DAHUTS - ANDAGE    6870.0
                      LES TUNIQUES BLEUES   1400.0
                                  EVASION   7110.0
                            LES AUBEPINES   4632.0
                              S.J.N. CLUB   1030.0
                            LES CARACOLES   5000.0
                       LES QUILLES LIBRES   6700.0
                             LA CLAIRIÈRE   1170.0
                               RJCV SPORT   5020.0
                          CLERLANDE SPORT   1340.0
                              LES OURSONS   5300.0
                              LES COPAINS   1083.0
                              LES GRINGOS   5550.0
                                   METHYS   4000.0
                                 L'

### 5.4 Impact on Use-Cases

Mapping of key quality issues to specific dashboard use-cases:

| Use-Case | Affected By | Severity |
|----------|-------------|----------|
| Distribution by age & gender | M1, M2, M3, L1 | 🟡 Medium |
| Average age of best performers | M2, C4 | 🟡 Medium |
| Athletes in multiple disciplines | C7, M7 | 🔴 High |
| Experience vs performance | C3, C4 | 🔴 High |
| Year-over-year development | C3, C7, M8 | 🔴 High |
| 3+ competitions → better performance? | C3, C4 | 🔴 High |
| Score improvements since previous edition | C3, C7 | 🔴 High |
| Best results | C3, C4 | 🔴 High |
| 20% disqualification rule | C4 (DQ parsing) | 🔴 High |
| Regional analysis | C2, C5, C6, M4 | 🔴 High |
| Multi-year athlete tracking | (Code join solid) | 🟢 Low |

### 5.5 Recommended ETL Transformation Priority

Ordered by **impact × dependency** — later steps depend on earlier ones.

| Priority | Task | Issues Resolved |
|----------|------|-----------------|
| **1** | Drop empty rows & filter Person types | C1, L3 |
| **2** | Country / Province / City normalization | C6, M4, M5, L2, L7 |
| **3** | Club name fuzzy matching | C2, M6 |
| **4** | `General` club recovery | C5, M10 |
| **5** | Sport & Event name canonicalization | C7, M7 |
| **6** | Score parsing | C3 |
| **7** | Place / Rank parsing & Medal derivation | C4 |
| **8** | DOB / Age cleanup | M1, M2, M3 |
| **9** | Gender standardization | L1 |
| **10** | Handle 2023 schema gap & 2025 missingness | M8, M9 |
| **11** | Certificate columns | L4 |

In [97]:
# Final summary statistics
print("=" * 80)
print("FINAL DATA QUALITY OVERVIEW")
print("=" * 80)
print(f"\n{'Source':<30} {'Rows':>10} {'Quality Note'}")
print("-" * 80)
print(f"{'Certifications':<30} {len(cert_df):>10,}   {empty_rows} empty rows removed; {dob_missing} DOBs missing")
print(f"{'Clubs':<30} {len(clubs_df):>10,}   {country_variants} country variants; {province_count} province variants")
print(f"{'Results (all years)':<30} {len(results_df):>10,}   {len(result_files)} files; {len(ci_no_match)} unmatched clubs")
print(f"\n{'Join':<35} {'Rate':>10}")
print("-" * 50)
print(f"{'Results.Code → Cert.Code':<35} {'99.6%':>10}")
print(f"{'Results.Club → Clubs.Name (exact)':<35} {f'{len(exact_match)/len(result_club_names)*100:.1f}%':>10}")
print(f"{'Results.Club → Clubs.Name (norm.)':<35} {f'{len(ci_match)/len(result_club_names)*100:.1f}%':>10}")
print(f"{'Cert.Club → Clubs.Name':<35} {f'{len(cert_clubs_matched)/len(cert_club_names)*100:.1f}%':>10}")
print(f"\nCritical issues: 7  |  Medium issues: 10  |  Low issues: 7")
print(f"High-severity use-case blocks: 8 out of 11 use-cases need C3 (score) or C4 (place) fixes")

FINAL DATA QUALITY OVERVIEW

Source                               Rows Quality Note
--------------------------------------------------------------------------------
Certifications                     20,221   780 empty rows removed; 1210 DOBs missing
Clubs                                 436   14 country variants; 24 province variants
Results (all years)               112,375   9 files; 112 unmatched clubs

Join                                      Rate
--------------------------------------------------
Results.Code → Cert.Code                 99.6%
Results.Club → Clubs.Name (exact)        75.5%
Results.Club → Clubs.Name (norm.)        77.5%
Cert.Club → Clubs.Name                   97.0%

Critical issues: 7  |  Medium issues: 10  |  Low issues: 7
High-severity use-case blocks: 8 out of 11 use-cases need C3 (score) or C4 (place) fixes
